In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# Hyperparameters
batch_size = 4     # num independent examples
block_size = 1024  # max sequence length
n_embd = 768       # total embedding dim, both in and out, divisible by n_head
n_head = 12        # number of heads
assert n_embd % n_head == 0
head_size = n_embd // n_head

# Init
torch.manual_seed(42)
c_attn_W = torch.randn(2304, 768) / 2304**0.5
c_attn_b = torch.randn(2304)
c_proj_W = torch.randn(768, 768) / 768**0.5
c_proj_b = torch.randn(768)
x = torch.randn(batch_size,block_size,n_embd)

In [3]:
# Reference Implementation

class CausalSelfAttentionMarcin(nn.Module):
    """Multiple self-attention heads"""
    def __init__(self, n_head, n_embd):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head

        self.c_attn = nn.Linear(n_embd, 3*n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1  # flag to scale proj into residual

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)  # B, T, nh*hs
        q = q.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        k = k.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        v = v.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        q = q.transpose(1, 2)  # B,nh,T,hs
        k = k.transpose(1, 2)  # B,nh,T,hs
        v = v.transpose(1, 2)  # B,nh,T,hs

        # W_affin = q @ k.mT / k.shape[-1]**0.5  # B,nh,T,hs @ B,nh,hs,T -> B,nh,T,T
        # W_affin = W_affin.masked_fill(self.bias[:,:,:T,:T]==0, float('-inf'))
        # W_affin = torch.softmax(W_affin, dim=-1)  # B,nh,T,T
        # y = W_affin @ v    # B,nh,T,T @ B,nh,T,hs -> B,nh,T,hs
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        y = y.transpose(1, 2)  # B,T,nh,hs
        y = y.contiguous()
        y = y.view(B,T,C)

        out = self.c_proj(y)
        return out

In [4]:
# Run reference implementation

csa_m = CausalSelfAttentionMarcin(n_head=n_head, n_embd=n_embd)
csa_m_state = csa_m.state_dict()
csa_m_state['c_attn.weight'] = c_attn_W.clone()
csa_m_state['c_attn.bias'] = c_attn_b.clone()
csa_m_state['c_proj.weight'] = c_proj_W.clone()
csa_m_state['c_proj.bias'] = c_proj_b.clone()
csa_m.load_state_dict(csa_m_state)

y_m2 = csa_m(x)
print(y_m2.shape)
print(y_m2.sum().item())

torch.Size([4, 1024, 768])
-18783.599609375


In [84]:
class CausalSelfAttentionRoPE(nn.Module):
    """Multiple self-attention heads"""
    def __init__(self, n_head, n_embd):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head

        self.c_q = nn.Linear(n_embd, n_embd)
        self.c_k = nn.Linear(n_embd, n_embd)
        self.c_v = nn.Linear(n_embd, n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)

    def _apply_rope(self, q, cos, sin):
        B, T, nh, hs = q.size()
        # Trim sin, cos to T and add batch dim
        sin = sin[:T, :].view(1, T, 1, hs//2)     # 1,T,1,hs/2
        cos = cos[:T, :].view(1, T, 1, hs//2)
        # Split x/y
        q_x, q_y = q[..., :hs//2], q[..., hs//2:]  # B,T,nh,hs/2
        # Apply rotation
        q_x_rot = cos * q_x - sin * q_y
        q_y_rot = sin * q_x + cos * q_y
        # Combine back
        q_rot = torch.cat([q_x_rot, q_y_rot], dim=-1)        # B,T,nh,hs
        return q_rot

    def forward(self, x, cos, sin):
        B, T, C = x.size()
        q = self.c_q(x)    # B, T, nh*hs
        k = self.c_k(x)    # B, T, nh*hs
        v = self.c_v(x)    # B, T, nh*hs
        q = q.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        k = k.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs
        v = v.view(B, T, self.n_head, C//self.n_head)  # B,T,nh,hs

        q_rot = self._apply_rope(q, cos, sin)
        k_rot = self._apply_rope(k, cos, sin)

        q = q_rot.transpose(1, 2)  # B,nh,T,hs
        k = k_rot.transpose(1, 2)  # B,nh,T,hs
        v = v.transpose(1, 2)  # B,nh,T,hs

        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        y = y.transpose(1, 2)  # B,T,nh,hs
        y = y.contiguous()
        y = y.view(B,T,C)

        out = self.c_proj(y)
        return out

In [85]:
def precalculate_cos_sin(seq_len, head_size, base=10_000):
    # Compute exponent for the RoPE frequencies
    theta = torch.arange(0, head_size, step=2)
    theta = base**-(theta/head_size)     # head_size//2
    pos = torch.arange(0, seq_len)       # seq_len
    tmp = torch.outer(pos, theta)        # seq_len, head_size//2
    sin, cos = torch.sin(tmp), torch.cos(tmp)
    return cos, sin

In [86]:
csa_m2 = CausalSelfAttentionRoPE(n_head=n_head, n_embd=n_embd)
csa_m2_state = csa_m2.state_dict()

csa_m2_state['c_q.weight'] = c_attn_W[:n_embd].clone()
csa_m2_state['c_q.bias'] = c_attn_b[:n_embd].clone()
csa_m2_state['c_k.weight'] = c_attn_W[n_embd:2*n_embd].clone()
csa_m2_state['c_k.bias'] = c_attn_b[n_embd:2*n_embd].clone()
csa_m2_state['c_v.weight'] = c_attn_W[2*n_embd:].clone()
csa_m2_state['c_v.bias'] = c_attn_b[2*n_embd:].clone()

csa_m2_state['c_proj.weight'] = c_proj_W.clone()
csa_m2_state['c_proj.bias'] = c_proj_b.clone()
csa_m2.load_state_dict(csa_m2_state)

cos, sin = precalculate_cos_sin(10*1024, head_size)  # overprovision to support longer sequences
print(sin.shape, cos.shape)

y_m2 = csa_m2(x, cos, sin)
print(y_m2.shape)
print(y_m2.sum().item())

torch.Size([10240, 32]) torch.Size([10240, 32])
torch.Size([4, 1024, 768])
-21446.939453125
